# Tweeting Fear — GPT-2 Fine-Tuning v2
### BUSN 20800 Big Data
**Hewitt Watkins · Erik Lopez · Mateo Fretes · Vedant Dangayach**

**Changes from v1:**
- Clean tweet text (URLs/media tokens stripped, ≥5 words)
- 6 LDA topics (vs 4)
- 5-bucket VIX/EPU conditioning (VERY_LOW → VERY_HIGH)
- Saves to `booth_results_v2/` in Drive

**Input:** `booth_final_data/tweet_labels_v2.csv`
**Output:** `booth_results_v2/` in Google Drive

> Run on a GPU runtime: Runtime → Change runtime type → T4 GPU

## 0. Mount Drive & Create Output Folders

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR    = '/content/drive/MyDrive/booth_final_data'
RESULTS_DIR = '/content/drive/MyDrive/booth_results_v2'
MODELS_DIR  = os.path.join(RESULTS_DIR, 'models', 'trump_gpt2_v2')
RESULTS_OUT = os.path.join(RESULTS_DIR, 'results')

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(RESULTS_OUT, exist_ok=True)

print('Drive mounted. Folders ready:')
print(f'  Data in:     {DATA_DIR}')
print(f'  Models out:  {MODELS_DIR}')
print(f'  Results out: {RESULTS_OUT}')

Mounted at /content/drive
Drive mounted. Folders ready:
  Data in:     /content/drive/MyDrive/booth_final_data
  Models out:  /content/drive/MyDrive/booth_results_v2/models/trump_gpt2_v2
  Results out: /content/drive/MyDrive/booth_results_v2/results


In [2]:
import zipfile

zip_path    = os.path.join(DATA_DIR, 'data.zip')
labels_path = os.path.join(DATA_DIR, 'tweet_labels_v2.csv')

if os.path.exists(labels_path):
    print(f'tweet_labels_v2.csv already present — ready.')

elif os.path.exists(os.path.join(DATA_DIR, 'data', 'tweet_labels_v2.csv')):
    DATA_DIR    = os.path.join(DATA_DIR, 'data')
    labels_path = os.path.join(DATA_DIR, 'tweet_labels_v2.csv')
    print(f'Found in data/ subfolder — updated DATA_DIR to {DATA_DIR}')

else:
    if not os.path.exists(zip_path):
        raise FileNotFoundError(
            f'tweet_labels_v2.csv not found in {DATA_DIR}\n'
            'Please upload tweet_labels_v2.csv (output of unsupervised_v2.ipynb) to booth_final_data/ in Drive.'
        )
    print(f'Unzipping {zip_path} ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)
    if os.path.exists(os.path.join(DATA_DIR, 'data', 'tweet_labels_v2.csv')):
        DATA_DIR    = os.path.join(DATA_DIR, 'data')
        labels_path = os.path.join(DATA_DIR, 'tweet_labels_v2.csv')
    if not os.path.exists(labels_path):
        raise FileNotFoundError('tweet_labels_v2.csv not found after unzip.')

print(f'Using: {labels_path}')

Found in data/ subfolder — updated DATA_DIR to /content/drive/MyDrive/booth_final_data/data
Using: /content/drive/MyDrive/booth_final_data/data/tweet_labels_v2.csv


## 1. Install & Import

In [3]:
!pip install transformers datasets accelerate -q
print('Done')

Done


In [4]:
import pandas as pd
import numpy as np
import torch
import json
import random
import math
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer,
    Trainer, TrainingArguments,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset as HFDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Change runtime to T4 GPU.')

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Memory: 102.0 GB


## 2. Load Data & Inspect Label Distribution

In [5]:
df = pd.read_csv(labels_path)
df = df.dropna(subset=['gpt2_prompt', 'vix_bucket', 'epu_bucket', 'lda_topic'])
df['lda_topic'] = df['lda_topic'].astype(int)
print(f'Loaded {len(df):,} labeled tweets')
print(f'Columns: {list(df.columns)}')
print()
print('VIX bucket counts:')
print(df['vix_bucket'].value_counts().sort_index().to_string())
print()
print('EPU bucket counts:')
print(df['epu_bucket'].value_counts().sort_index().to_string())
print()
print('Topic counts:')
print(df['lda_topic'].value_counts().sort_index().to_string())
print()
combo = df.groupby(['vix_bucket','epu_bucket','lda_topic']).size().reset_index(name='count')
print(f'Total conditioning combinations: {len(combo)}')
print(f'Sparsest combination: {combo["count"].min()} tweets')
print(f'Richest  combination: {combo["count"].max()} tweets')

Loaded 26,627 labeled tweets
Columns: ['tweet_idx', 'date', 'text', 'week_end', 'lda_topic', 'kmeans_cluster', 'word_count', 'vix_bucket', 'epu_bucket', 'vix', 'epu', 'gpt2_prompt']

VIX bucket counts:
vix_bucket
VIX_HIGH         5430
VIX_LOW          5700
VIX_MED          5217
VIX_VERY_HIGH    5588
VIX_VERY_LOW     4692

EPU bucket counts:
epu_bucket
EPU_HIGH         4788
EPU_LOW          5488
EPU_MED          4686
EPU_VERY_HIGH    5992
EPU_VERY_LOW     5673

Topic counts:
lda_topic
0    6747
1    4913
2    3330
3    2140
4    6463
5    3034

Total conditioning combinations: 137
Sparsest combination: 1 tweets
Richest  combination: 991 tweets


In [6]:
# Load cutoffs from v2 file, fall back to computing from data
cutoffs_path = os.path.join(DATA_DIR, 'bucket_cutoffs_v2.json')
if os.path.exists(cutoffs_path):
    with open(cutoffs_path) as f:
        cutoffs = json.load(f)
    print('Loaded bucket_cutoffs_v2.json')
else:
    # Recompute from weekly data in tweet_labels_v2.csv
    weekly = df.groupby('week_end')[['vix','epu']].first().dropna()
    vix_cuts = weekly['vix'].quantile([0.2, 0.4, 0.6, 0.8]).values
    epu_cuts = weekly['epu'].quantile([0.2, 0.4, 0.6, 0.8]).values
    cutoffs = {'vix': vix_cuts.tolist(), 'epu': epu_cuts.tolist(), 'n_bins': 5}
    print('Computed cutoffs from data (bucket_cutoffs_v2.json not found)')

with open(os.path.join(RESULTS_OUT, 'bucket_cutoffs_v2.json'), 'w') as f:
    json.dump(cutoffs, f, indent=2)

print(f'VIX cutoffs: {[round(x,2) for x in cutoffs["vix"]]}')
print(f'EPU cutoffs: {[round(x,2) for x in cutoffs["epu"]]}')
print('Saved bucket_cutoffs_v2.json to results/')

Loaded bucket_cutoffs_v2.json
VIX cutoffs: [12.63, 14.91, 17.54, 22.66]
EPU cutoffs: [108.21, 124.94, 149.97, 218.06]
Saved bucket_cutoffs_v2.json to results/


## 3. Tokenizer & Model Setup

5-bucket VIX/EPU special tokens + 6 topic tokens.

In [7]:
MODEL_NAME = 'gpt2'
tokenizer  = GPT2Tokenizer.from_pretrained(MODEL_NAME)

topics = sorted(df['lda_topic'].unique())
print(f'Topics found: {topics}')

VIX_LABELS = ['VIX_VERY_LOW', 'VIX_LOW', 'VIX_MED', 'VIX_HIGH', 'VIX_VERY_HIGH']
EPU_LABELS = ['EPU_VERY_LOW', 'EPU_LOW', 'EPU_MED', 'EPU_HIGH', 'EPU_VERY_HIGH']

special_tokens = VIX_LABELS + EPU_LABELS + [f'[TOPIC_{i}]' for i in topics]
tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})
tokenizer.pad_token = tokenizer.eos_token

print(f'Special tokens added: {special_tokens}')
print(f'Vocabulary size: {len(tokenizer):,}')

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
model.to(device)

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {params/1e6:.1f}M')
print('Model ready.')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Topics found: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Special tokens added: ['VIX_VERY_LOW', 'VIX_LOW', 'VIX_MED', 'VIX_HIGH', 'VIX_VERY_HIGH', 'EPU_VERY_LOW', 'EPU_LOW', 'EPU_MED', 'EPU_HIGH', 'EPU_VERY_HIGH', '[TOPIC_0]', '[TOPIC_1]', '[TOPIC_2]', '[TOPIC_3]', '[TOPIC_4]', '[TOPIC_5]']
Vocabulary size: 50,273


...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model parameters: 124.5M
Model ready.


## 4. Prepare Dataset

90/10 split. Format: `[VIX_X] [EPU_X] [TOPIC_N] <tweet><|endoftext|>`

In [8]:
df['training_text'] = df['gpt2_prompt'].astype(str) + tokenizer.eos_token

random.seed(42)
indices   = list(range(len(df)))
random.shuffle(indices)
split_idx = int(0.9 * len(indices))
train_idx = indices[:split_idx]
test_idx  = indices[split_idx:]

train_texts = df.iloc[train_idx]['training_text'].tolist()
test_texts  = df.iloc[test_idx]['training_text'].tolist()
print(f'Train: {len(train_texts):,}  |  Test: {len(test_texts):,}')

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

train_ds = HFDataset.from_dict({'text': train_texts}).map(
    tokenize_fn, batched=True, remove_columns=['text'])
test_ds  = HFDataset.from_dict({'text': test_texts}).map(
    tokenize_fn, batched=True, remove_columns=['text'])

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print('Datasets ready.')

Train: 23,964  |  Test: 2,663


Map:   0%|          | 0/23964 [00:00<?, ? examples/s]

Map:   0%|          | 0/2663 [00:00<?, ? examples/s]

Datasets ready.


## 5. Fine-Tune

In [9]:
training_args = TrainingArguments(
    output_dir=os.path.join(RESULTS_DIR, 'models', 'checkpoints_v2'),
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=5e-5,
    fp16=(device.type == 'cuda'),
    logging_dir=os.path.join(RESULTS_OUT, 'logs'),
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    save_total_limit=2,
    report_to='none',
    prediction_loss_only=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
)

print('Starting training...')
train_result = trainer.train()
print('Training complete.')
print(f'  Total steps: {train_result.global_step}')
print(f'  Train loss:  {train_result.training_loss:.4f}')

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting training...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.724481,2.627376
2,2.491018,2.553435
3,2.392498,2.536921


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Training complete.
  Total steps: 8988
  Train loss:  2.6171


## 6. Save Model & Tokenizer to Drive

In [10]:
model.save_pretrained(MODELS_DIR)
tokenizer.save_pretrained(MODELS_DIR)
print(f'Model saved to {MODELS_DIR}')

metrics = {
    'train_loss':    float(train_result.training_loss),
    'global_steps':  int(train_result.global_step),
    'train_samples': int(len(train_texts)),
    'test_samples':  int(len(test_texts)),
    'topics':        [int(t) for t in topics],
    'vix_labels':    VIX_LABELS,
    'epu_labels':    EPU_LABELS,
    'n_buckets':     5,
}
with open(os.path.join(RESULTS_OUT, 'training_metrics_v2.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved training_metrics_v2.json')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/booth_results_v2/models/trump_gpt2_v2
Saved training_metrics_v2.json


## 7. Evaluate — Perplexity

In [11]:
eval_results = trainer.evaluate()
perplexity   = math.exp(eval_results['eval_loss'])
print(f'Eval loss:   {eval_results["eval_loss"]:.4f}')
print(f'Perplexity:  {perplexity:.2f}')

with open(os.path.join(RESULTS_OUT, 'perplexity_v2.txt'), 'w') as f:
    f.write(f'eval_loss:  {eval_results["eval_loss"]:.4f}\n')
    f.write(f'perplexity: {perplexity:.2f}\n')
print('Saved perplexity_v2.txt')

Eval loss:   2.5369
Perplexity:  12.64
Saved perplexity_v2.txt


## 8. Generate Sample Tweets (B2 — All Topics per VIX×EPU)

All 25 VIX×EPU combinations × 6 topics = 150 sample tweets.

In [12]:
model.eval()

def generate_tweet(vix_label, epu_label, topic_id,
                   max_new_tokens=80, temperature=0.85, top_p=0.92):
    prompt     = f'[{vix_label}] [{epu_label}] [TOPIC_{topic_id}]'
    input_ids  = tokenizer.encode(prompt, return_tensors='pt').to(device)
    prompt_len = input_ids.shape[1]
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][prompt_len:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

sample_rows = []
print('Generating sample tweets for all VIX x EPU x Topic combinations...')
for vl in VIX_LABELS:
    for el in EPU_LABELS:
        for t in topics:
            text = generate_tweet(vl, el, t)
            sample_rows.append({
                'vix_label': vl, 'epu_label': el,
                'topic': t, 'generated_tweet': text
            })
        print(f'  {vl} x {el}: done')

samples_df = pd.DataFrame(sample_rows)
samples_df.to_csv(os.path.join(RESULTS_OUT, 'sample_generations_v2.csv'), index=False)
print(f'Saved sample_generations_v2.csv — {len(samples_df)} generated tweets')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generating sample tweets for all VIX x EPU x Topic combinations...
  VIX_VERY_LOW x EPU_VERY_LOW: done
  VIX_VERY_LOW x EPU_LOW: done
  VIX_VERY_LOW x EPU_MED: done
  VIX_VERY_LOW x EPU_HIGH: done
  VIX_VERY_LOW x EPU_VERY_HIGH: done
  VIX_LOW x EPU_VERY_LOW: done
  VIX_LOW x EPU_LOW: done
  VIX_LOW x EPU_MED: done
  VIX_LOW x EPU_HIGH: done
  VIX_LOW x EPU_VERY_HIGH: done
  VIX_MED x EPU_VERY_LOW: done
  VIX_MED x EPU_LOW: done
  VIX_MED x EPU_MED: done
  VIX_MED x EPU_HIGH: done
  VIX_MED x EPU_VERY_HIGH: done
  VIX_HIGH x EPU_VERY_LOW: done
  VIX_HIGH x EPU_LOW: done
  VIX_HIGH x EPU_MED: done
  VIX_HIGH x EPU_HIGH: done
  VIX_HIGH x EPU_VERY_HIGH: done
  VIX_VERY_HIGH x EPU_VERY_LOW: done
  VIX_VERY_HIGH x EPU_LOW: done
  VIX_VERY_HIGH x EPU_MED: done
  VIX_VERY_HIGH x EPU_HIGH: done
  VIX_VERY_HIGH x EPU_VERY_HIGH: done
Saved sample_generations_v2.csv — 150 generated tweets


## 9. Preview Generated Tweets

In [13]:
for vl in VIX_LABELS:
    for el in EPU_LABELS:
        subset = samples_df[(samples_df['vix_label']==vl) & (samples_df['epu_label']==el)]
        print(f'\n{"="*60}')
        print(f'  [{vl}] [{el}]')
        print(f'{"="*60}')
        for _, row in subset.iterrows():
            print(f'  [Topic {row["topic"]}] {row["generated_tweet"][:140]}')


  [VIX_VERY_LOW] [EPU_VERY_LOW]
  [Topic 0] “The Do Nothing Democrats, led by Adam Schiff and others, pushed back yesterday with a barrage of accusations that the President was not inf
  [Topic 1] ....and he, in fact, should not be allowed to act in that manner. His only crime is not to use force, and his only crime should be to seek h
  [Topic 2] This morning, I joined hundreds of incredible supporters of the #USArmy at the @WhiteHouse to remember those who have served our Country wit
  [Topic 3] ....John has been a tremendous champion for our Great Farmers, Military, and Vets. He has been with us from the very beginning, and he will 
  [Topic 4] The China Virus continues to spread, especially in China. I have instructed the Centers to keep a full & complete list of all deaths and ill
  [Topic 5] A must watch. Will be on @foxandfriends at 10:00 A.M. Enjoy! ….@foxandwhatsnewt. Enjoy!!! DJT @FoxNews ….DonaldJTrump.com/Polls?p=116002 . D

  [VIX_VERY_LOW] [EPU_LOW]
  [Topic 0] I have ne

## 10. Memorization Check

In [14]:
import difflib

train_originals = df['gpt2_prompt'].astype(str).tolist()

def most_similar(generated_text):
    scores = [
        difflib.SequenceMatcher(None, generated_text.lower(), t.lower()).ratio()
        for t in train_originals
    ]
    best_idx = max(range(len(scores)), key=lambda i: scores[i])
    return scores[best_idx], train_originals[best_idx]

THRESHOLD = 0.85
flagged = 0
results = []
print(f'Memorization check (flag threshold: {THRESHOLD})\n')

for _, row in samples_df.iterrows():
    score, match = most_similar(row['generated_tweet'])
    flag = score > THRESHOLD
    if flag:
        flagged += 1
    results.append({'vix_label': row['vix_label'], 'epu_label': row['epu_label'],
                    'topic': row['topic'], 'similarity': round(score, 4),
                    'flagged': flag, 'nearest_training_tweet': match})
    marker = '  ⚠️  SUSPICIOUS' if flag else ''
    print(f'[{row["vix_label"]}][{row["epu_label"]}][Topic {row["topic"]}]  sim={score:.3f}{marker}')
    if flag:
        print(f'  Generated: {row["generated_tweet"][:120]}')
        print(f'  Nearest:   {match[:120]}\n')

print(f'\n{flagged}/{len(samples_df)} flagged  |  mean similarity: {sum(r["similarity"] for r in results)/len(results):.3f}')

pd.DataFrame(results).to_csv(os.path.join(RESULTS_OUT, 'memorization_check_v2.csv'), index=False)
print('Saved memorization_check_v2.csv')

Memorization check (flag threshold: 0.85)

[VIX_VERY_LOW][EPU_VERY_LOW][Topic 0]  sim=0.330
[VIX_VERY_LOW][EPU_VERY_LOW][Topic 1]  sim=0.356
[VIX_VERY_LOW][EPU_VERY_LOW][Topic 2]  sim=0.391
[VIX_VERY_LOW][EPU_VERY_LOW][Topic 3]  sim=0.321
[VIX_VERY_LOW][EPU_VERY_LOW][Topic 4]  sim=0.324
[VIX_VERY_LOW][EPU_VERY_LOW][Topic 5]  sim=0.389
[VIX_VERY_LOW][EPU_LOW][Topic 0]  sim=0.355
[VIX_VERY_LOW][EPU_LOW][Topic 1]  sim=0.352
[VIX_VERY_LOW][EPU_LOW][Topic 2]  sim=0.355
[VIX_VERY_LOW][EPU_LOW][Topic 3]  sim=0.539
[VIX_VERY_LOW][EPU_LOW][Topic 4]  sim=0.350
[VIX_VERY_LOW][EPU_LOW][Topic 5]  sim=0.356
[VIX_VERY_LOW][EPU_MED][Topic 0]  sim=0.348
[VIX_VERY_LOW][EPU_MED][Topic 1]  sim=0.353
[VIX_VERY_LOW][EPU_MED][Topic 2]  sim=0.384
[VIX_VERY_LOW][EPU_MED][Topic 3]  sim=0.340
[VIX_VERY_LOW][EPU_MED][Topic 4]  sim=0.393
[VIX_VERY_LOW][EPU_MED][Topic 5]  sim=0.405
[VIX_VERY_LOW][EPU_HIGH][Topic 0]  sim=0.373
[VIX_VERY_LOW][EPU_HIGH][Topic 1]  sim=0.374
[VIX_VERY_LOW][EPU_HIGH][Topic 2]  sim=0.355
